# Creating Tax Assessments with Feature Engineering
## Solution

**Short name (GitHub):** `TaxCreate`

Worked answers for `TaxCreate_Practice_Skeleton.ipynb`. Numbers from `random_state=42`, default `RandomForestRegressor` (100 trees), leakage-safe encoding.

Hold-out (184 of 920): **MAE ≈ $1,149**, **RMSE ≈ $1,703**, **R² ≈ 0.923**. Mean-predictor baseline MAE ≈ **$4,849**. LinearRegression MAE ≈ **$1,478**, R² ≈ **0.893**. Train R² ≈ 0.991.


## Inline cheat-sheet

| After clean | Role on this card |
|-------------|-------------------|
| `gross_income` | strongest numeric corr with `tax_due` (0.84); impurity ~0.75 |
| `deduction_usd` / `credits_usd` | negative corr; next impurity block |
| `business_usd` / `wages_usd` / `capgains_usd` | components of gross; still useful after the composite exists |
| `state_rate` | 0–7% statutory add-on; visible in the boxplot |
| `dependents` | lowers the bill (exemption-style) |
| high-card encode | `tax_office` (8), `income_class` (8), `filing_status` (5) |
| one-hot | channel, deduction type, complexity, timing, digital, preparer, resident |


## Flowchart

![flow](taxcreate_flowchart.png)


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")


## 1. Load


In [ ]:
df = pd.read_csv("data/tax_returns.csv")
print(df.head())
print(df.shape)
print(df.info())
print(df.nunique())
print(df["tax_due"].describe())
print("zero share", (df["tax_due"] == 0).mean())


## 2. Money + gross


In [ ]:
money_cols = ["wages_usd", "business_usd", "capgains_usd", "deduction_usd", "credits_usd"]
for col in money_cols:
    df[col] = (
        df[col].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )
df["gross_income"] = df["wages_usd"] + df["business_usd"] + df["capgains_usd"]
print(df[money_cols + ["gross_income"]].head())


## 3. Rate, stars, year, dependents, pages


In [ ]:
df["state_rate"] = (
    df["state_rate"].astype(str).str.replace("%", "", regex=False).astype(float) / 100
)
df["compliance_stars"] = (
    df["compliance_stars"].astype(str)
    .str.replace(" stars", "", regex=False)
    .str.replace(" star", "", regex=False)
    .astype(int)
)
df["tax_year_n"] = (
    df["tax_year"].astype(str)
    .str.replace("Not Available", "0", regex=False)
    .str.replace(" TY", "", regex=False)
    .astype(int)
)
df = df.drop(columns=["tax_year"])
df["dependents"] = df["dependents"].astype(str).str.extract(r"(\d+)", expand=False).astype(int)
df["return_pages"] = df["return_pages"].astype(str).str.replace("-page", "", regex=False).astype(int)
print(df[["state_rate", "compliance_stars", "tax_year_n", "dependents", "return_pages"]].head())


## 4. EDA


In [ ]:
numeric_df = df.select_dtypes(include="number")
print(numeric_df.corr()["tax_due"].sort_values(ascending=False).round(3))

plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap of numeric tax features")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x="dependents", y="tax_due", data=df)
plt.title("tax_due by dependents"); plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x="state_rate", y="tax_due", data=df)
plt.title("tax_due by state rate"); plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(df["tax_due"], bins=40, kde=True)
plt.axvline(df["tax_due"].median(), color="crimson", ls="--", label="median")
plt.axvline(df["tax_due"].mean(), color="navy", ls=":", label="mean")
plt.title("tax_due is right-skewed with a zero mass")
plt.legend(); plt.show()


Expected read: `gross_income` 0.84 leads. Workpaper *counts* also look strong (0.71) because they scale with income — they are not a filing input. Credits and deductions correlate negatively. About 16% of assessments are $0 after credits.


## 5. Encode + split (lesson version)


In [ ]:
work = df.copy()
cat_cols = work.select_dtypes(include=["object"]).columns.tolist()
print({c: work[c].nunique() for c in cat_cols})
for col in cat_cols:
    if work[col].nunique() < 5:
        dummies = pd.get_dummies(work[col], prefix=col, drop_first=True)
        work = pd.concat([work, dummies], axis=1)
        work.drop(columns=[col], inplace=True)
    else:
        work[col] = work[col].map(work.groupby(col)["tax_due"].mean())
X = work.drop(columns=["tax_due"]); y = work["tax_due"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X.shape, X_train.shape, X_test.shape)


## 6. Forest


In [ ]:
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
print("train R²", round(r2_score(y_train, rf_model.predict(X_train)), 4))


## 7. Evaluate


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
base_mae = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float))
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²: {r2:.4f}")
print(f"mean-baseline MAE: {base_mae:,.2f}")

importances = rf_model.feature_importances_
names = np.array(X_train.columns)
top = np.argsort(importances)[-5:][::-1]
print(pd.Series(importances[top], index=names[top]))

plt.figure(figsize=(8, 5))
sns.barplot(x=importances[top], y=names[top], color="#1F4E79")
plt.title("Top 5 impurity importances"); plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, s=28, alpha=0.55)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([lo, hi], [lo, hi], "r--")
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.title(f"Hold-out  ·  MAE={mae:,.0f}  R²={r2:.3f}")
plt.show()


### Reference numbers (rs=42, leakage-safe encode on the same card)

| Metric | Value |
|--------|-------|
| Test MAE | **$1,149** |
| Test RMSE | **$1,703** |
| Test R² | **0.923** |
| Train R² | 0.991 |
| Mean baseline MAE | **$4,849** |
| LinearRegression | MAE $1,478 / R² 0.893 |
| Ridge(α=1) | MAE $1,475 / R² 0.893 |

The forest beats the mean guess by about 4× and beats linear by a few hundred dollars. Linear is already strong because the assessment is a noisy statutory formula of the parsed dollar fields — that is the point of a tax-*creation* card.

Top impurity: `gross_income` (~0.75), then `deduction_usd`, `business_usd`, `credits_usd`. Permutation on the hold-out keeps the same block and lifts `state_rate` above the income components.


## 8. Alternates


In [ ]:
raw = pd.read_csv("data/tax_returns.csv")
raw["wages_usd"] = raw["wages_usd"].astype(str).str.replace(r"[^0-9.]", "", regex=True).astype(float)
print(raw["wages_usd"].head())

y_safe = df["tax_due"]
X_raw = df.drop(columns=["tax_due"])
Xtr, Xte, ytr, yte = train_test_split(X_raw, y_safe, test_size=0.2, random_state=42)
cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
low = [c for c in cat_cols if df[c].nunique() < 5]
high = [c for c in cat_cols if df[c].nunique() >= 5]
Xtr_enc, Xte_enc = Xtr.copy(), Xte.copy()
for col in high:
    means = pd.concat([Xtr_enc[col], ytr], axis=1).groupby(col)[ytr.name].mean()
    Xtr_enc[col] = Xtr_enc[col].map(means)
    Xte_enc[col] = Xte_enc[col].map(means).fillna(ytr.mean())
for col in low:
    dtr = pd.get_dummies(Xtr_enc[col], prefix=col, drop_first=True)
    dte = pd.get_dummies(Xte_enc[col], prefix=col, drop_first=True)
    dte = dte.reindex(columns=dtr.columns, fill_value=0)
    Xtr_enc = pd.concat([Xtr_enc.drop(columns=[col]), dtr], axis=1)
    Xte_enc = pd.concat([Xte_enc.drop(columns=[col]), dte], axis=1)

rf_safe = RandomForestRegressor(random_state=42)
rf_safe.fit(Xtr_enc, ytr)
yp_safe = rf_safe.predict(Xte_enc)
print("SAFE MAE", round(mean_absolute_error(yte, yp_safe), 2), "R²", round(r2_score(yte, yp_safe), 4))

lr = LinearRegression().fit(Xtr_enc, ytr)
rg = Ridge(alpha=1.0).fit(Xtr_enc, ytr)
print("LR   ", round(mean_absolute_error(yte, lr.predict(Xte_enc)), 2), round(r2_score(yte, lr.predict(Xte_enc)), 4))
print("Ridge", round(mean_absolute_error(yte, rg.predict(Xte_enc)), 2), round(r2_score(yte, rg.predict(Xte_enc)), 4))

perm = permutation_importance(rf_safe, Xte_enc, yte, n_repeats=8, random_state=42)
print(pd.Series(perm.importances_mean, index=Xte_enc.columns).sort_values(ascending=False).head(8).round(4))


## 9. Practice sketches


In [ ]:
rf_log = RandomForestRegressor(random_state=42)
rf_log.fit(Xtr_enc, np.log1p(ytr))
yp_log = np.expm1(rf_log.predict(Xte_enc))
print("log-target MAE", round(mean_absolute_error(yte, yp_log), 2), "R²", round(r2_score(yte, yp_log), 4))

drop_act = [c for c in Xtr_enc.columns if "Number of" in c]
rf_np = RandomForestRegressor(random_state=42)
rf_np.fit(Xtr_enc.drop(columns=drop_act), ytr)
print("no-activity MAE", round(mean_absolute_error(yte, rf_np.predict(Xte_enc.drop(columns=drop_act))), 2),
      "R²", round(r2_score(yte, rf_np.predict(Xte_enc.drop(columns=drop_act))), 4))

taxable = (Xte_enc["gross_income"] - Xte_enc["deduction_usd"] - 1800 * Xte_enc["dependents"]).clip(lower=0)
# crude 18% blended statutory rebuild minus credits
stat = (0.18 * taxable + 5000 * Xte_enc.get("state_rate", 0) * taxable / taxable.replace(0, np.nan).fillna(1)
        - Xte_enc["credits_usd"]).clip(lower=0)
# simpler rebuild:
stat = (0.18 * taxable + Xte_enc["state_rate"] * taxable - Xte_enc["credits_usd"]).clip(lower=0)
print("blended-rate formula MAE", round(mean_absolute_error(yte, stat), 2))

tail = yte >= 5000
print("tail share", float(tail.mean()), "tail MAE", round(mean_absolute_error(yte[tail], yp_safe[tail]), 2))


## 10. Simulation


In [ ]:
N_EST = 100
MAX_DEPTH = None
NOISE_SD = 0
SUBSAMPLE = 1.0
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n = max(int(len(Xtr_enc) * SUBSAMPLE), 20)
idx = rng.choice(len(Xtr_enc), size=n, replace=False)
X_s = Xtr_enc.iloc[idx]
y_s = ytr.iloc[idx].astype(float) + rng.normal(0, NOISE_SD, size=n)
sim = RandomForestRegressor(n_estimators=N_EST, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
sim.fit(X_s, y_s)
print("sim MAE", round(mean_absolute_error(yte, sim.predict(Xte_enc)), 2),
      "R²", round(r2_score(yte, sim.predict(Xte_enc)), 4))

rows = []
for n_est in [10, 25, 50, 100, 200]:
    m = RandomForestRegressor(n_estimators=n_est, random_state=42)
    m.fit(Xtr_enc, ytr)
    p = m.predict(Xte_enc)
    rows.append((n_est, mean_absolute_error(yte, p), r2_score(yte, p)))
print(pd.DataFrame(rows, columns=["n_estimators", "MAE", "R2"]))


On this card the forest is already flat by ~25 trees (MAE $1,113–$1,225). Depth caps around 4 underfit the brackets. Adding $1,500 of label noise or cutting the train set to 40% both push MAE toward $1.6k–$2k. The parsed dollar fields carry the signal; extra trees do not invent a rate table.


## 11. Can / cannot

$1,149 MAE is about **18% of mean tax_due** and **24% of the median**. Useful as a desk *pre-check* on a draft extract (“does this assessment look like the rest of the book?”). Not a substitute for the published brackets — linear already gets most of the way there once `$` and `%` are gone.

`Number of Workpapers` looks important in a raw correlation and almost disappears under permutation. Drop it when the question is “create the assessment from the return,” not “how busy was the file.”
